<a href="https://colab.research.google.com/github/VintaBytes/Ciencia-de-datos/blob/main/Ciencia%20De%20Datos%20Con%20Python/CuadernosColab/CienciaDeDatos_Cap%C3%ADtulo007_Combinar_condiciones_en_filtros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Capítulo 7 — Combinar condiciones en filtros

## Breve repaso

En el trabajo anterior sobre filtros aprendimos a seleccionar filas de un `DataFrame` usando condiciones simples.

Vimos que una condición, como `df["precio"] > 1000`, genera una máscara booleana: una serie de valores `True` y `False` que indica qué filas cumplen la condición y cuáles no. Luego usamos esa máscara para obtener solamente las filas que nos interesaban.

También filtramos columnas numéricas, categóricas y de texto. Por ejemplo, seleccionamos ventas según su precio, sucursal, categoría o producto. Además, combinamos filtros con selección de columnas usando `loc`, lo que nos permitió obtener resultados más enfocados.

En este capítulo vamos a avanzar un paso más: vamos a combinar varias condiciones dentro de un mismo filtro.

Esto nos permitirá responder preguntas más precisas, como:

```text
¿Qué ventas de Tecnología tuvieron un precio mayor que 8000?
¿Qué ventas de la sucursal Centro se pagaron con crédito?
¿Qué productos que no son de Librería tuvieron pocas unidades vendidas?
¿Qué ventas corresponden a la sucursal Centro o a la sucursal Norte?
```

Para construir este tipo de filtros vamos a usar operadores lógicos:

```text
&   para expresar "y"
|   para expresar "o"
~   para expresar "no"
```

También vamos a ver por qué, al combinar condiciones en Pandas, es necesario encerrar cada condición entre paréntesis.

Al finalizar este capítulo deberías poder:

* Combinar dos o más condiciones dentro de un filtro.
* Usar `&` para seleccionar filas que cumplen dos condiciones al mismo tiempo.
* Usar `|` para seleccionar filas que cumplen al menos una de varias condiciones.
* Usar `~` para negar una condición.
* Comprender por qué los paréntesis son necesarios al combinar condiciones.
* Guardar condiciones en variables para escribir filtros más claros.
* Combinar filtros con selección de columnas usando `loc`.
* Reconocer errores frecuentes al combinar condiciones.

## Dataset de trabajo

Para este capítulo vamos a usar un dataset de ventas de una tienda escolar. El contexto es parecido al que veníamos usando, pero ahora incorporamos más productos, más categorías y una nueva columna llamada `forma_pago`. Esto nos permitirá construir filtros combinados más interesantes.

Cada fila representa una venta registrada. Las columnas indican el producto vendido, su categoría, el precio unitario, la cantidad vendida, la sucursal, la forma de pago y la fecha de la venta.

In [ ]:
import pandas as pd

datos = {
    "producto": [
        "Cuaderno", "Lapicera", "Mochila", "Regla", "Cartuchera",
        "Calculadora", "Auriculares", "Resaltadores", "Guardapolvo", "Compas",
        "Pendrive", "Carpeta", "Goma", "Lapiz", "Agenda"
    ],
    "categoria": [
        "Libreria", "Libreria", "Accesorios", "Libreria", "Accesorios",
        "Tecnologia", "Tecnologia", "Libreria", "Indumentaria", "Libreria",
        "Tecnologia", "Libreria", "Libreria", "Libreria", "Libreria"
    ],
    "precio": [
        1200, 350, 8500, 500, 3200,
        18500, 7600, 2100, 14500, 2600,
        9800, 950, 300, 250, 4300
    ],
    "cantidad_vendida": [
        15, 40, 5, 25, 8,
        3, 6, 12, 4, 10,
        7, 18, 30, 50, 9
    ],
    "sucursal": [
        "Centro", "Centro", "Norte", "Sur", "Norte",
        "Centro", "Sur", "Centro", "Norte", "Sur",
        "Centro", "Norte", "Sur", "Centro", "Sur"
    ],
    "forma_pago": [
        "Efectivo", "Debito", "Credito", "Efectivo", "Debito",
        "Credito", "Credito", "Debito", "Credito", "Efectivo",
        "Debito", "Efectivo", "Efectivo", "Debito", "Credito"
    ],
    "fecha": [
        "2024-03-01", "2024-03-01", "2024-03-02", "2024-03-02", "2024-03-03",
        "2024-03-03", "2024-03-04", "2024-03-04", "2024-03-05", "2024-03-05",
        "2024-03-06", "2024-03-06", "2024-03-07", "2024-03-07", "2024-03-08"
    ]
}

df = pd.DataFrame(datos)

df

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
0,Cuaderno,Libreria,1200,15,Centro,Efectivo,2024-03-01
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01
2,Mochila,Accesorios,8500,5,Norte,Credito,2024-03-02
3,Regla,Libreria,500,25,Sur,Efectivo,2024-03-02
4,Cartuchera,Accesorios,3200,8,Norte,Debito,2024-03-03
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
6,Auriculares,Tecnologia,7600,6,Sur,Credito,2024-03-04
7,Resaltadores,Libreria,2100,12,Centro,Debito,2024-03-04
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05
9,Compas,Libreria,2600,10,Sur,Efectivo,2024-03-05


El dataset tiene 15 filas y 7 columnas. Antes de construir filtros combinados, conviene observar qué columnas tenemos disponibles y qué tipo de preguntas podríamos hacer con ellas. Por ejemplo, ahora podemos combinar condiciones sobre `categoria`, `precio`, `cantidad_vendida`, `sucursal` y `forma_pago`.

Esa combinación de criterios es lo que nos permitirá construir consultas más precisas.

## Recordar un filtro simple

Antes de combinar condiciones, recordemos cómo se construye un filtro simple.

Si queremos seleccionar solamente las ventas de la categoría `"Tecnologia"`, podemos escribir:

In [ ]:
df[df["categoria"] == "Tecnologia"]

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
6,Auriculares,Tecnologia,7600,6,Sur,Credito,2024-03-04
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06


La condición:

```python
df["categoria"] == "Tecnologia"
```

genera una máscara booleana. Esa máscara indica, fila por fila, si la venta pertenece o no a la categoría `"Tecnologia"`.

También podemos filtrar por precio. Por ejemplo, si queremos seleccionar las ventas con precio mayor que `8000`, escribimos:

In [ ]:
df[df["precio"] > 8000]

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
2,Mochila,Accesorios,8500,5,Norte,Credito,2024-03-02
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06


En ambos casos usamos una sola condición. El primer filtro selecciona filas según una categoría. El segundo selecciona filas según un valor numérico. Pero muchas preguntas reales requieren más de una condición. Por ejemplo, podríamos querer seleccionar solamente las ventas de `"Tecnologia"` cuyo precio sea mayor que `8000`.

Para responder esa pregunta necesitamos combinar condiciones.

## Combinar condiciones con &

Cuando queremos que dos condiciones se cumplan al mismo tiempo, usamos el operador `&`. Por ejemplo, supongamos que queremos seleccionar las ventas que cumplen estas dos condiciones:

```text
la categoría es Tecnologia
el precio es mayor que 8000
```

Primero podemos mirar cada condición por separado.

In [ ]:
df["categoria"] == "Tecnologia"

,categoria
0,False
1,False
2,False
3,False
4,False
5,True
6,True
7,False
8,False
9,False


In [ ]:
df["precio"] > 8000

,precio
0,False
1,False
2,True
3,False
4,False
5,True
6,False
7,False
8,True
9,False


Cada una de esas expresiones produce una máscara booleana. Para filtrar las filas que cumplen ambas condiciones al mismo tiempo, debemos combinar esas máscaras usando `&`.

En Pandas, cuando combinamos condiciones, cada condición debe ir encerrada entre paréntesis.

In [ ]:
df[(df["categoria"] == "Tecnologia") & (df["precio"] > 8000)]

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06


El resultado muestra solamente las filas donde se cumplen las dos condiciones al mismo tiempo. Es decir, Pandas conserva las ventas que pertenecen a la categoría `"Tecnologia"` y que, además, tienen un precio mayor que `8000`.

Podemos leer la instrucción de esta manera:

```text
mostrá las filas donde
categoria sea igual a Tecnologia
y
precio sea mayor que 8000
```

El operador `&` funciona como un “y lógico”. Para que una fila aparezca en el resultado, ambas condiciones deben ser verdaderas.

## Por qué necesitamos paréntesis

Al combinar condiciones en Pandas, cada condición debe escribirse entre paréntesis. La forma correcta es:

```python
df[(df["categoria"] == "Tecnologia") & (df["precio"] > 8000)]
```

Los paréntesis le indican a Python qué comparación debe resolver primero. Sin ellos, la expresión puede interpretarse de una forma distinta a la que esperamos y producir un error.

Por ejemplo, esta forma no es recomendable:


In [ ]:
# Esta instrucción produciría un error porque faltan paréntesis.
# df[df["categoria"] == "Tecnologia" & df["precio"] > 8000]

El problema es que Python no interpreta automáticamente la expresión como “categoría igual a Tecnología y precio mayor que 8000”. Al faltar los paréntesis, intenta resolver partes de la expresión en un orden que no corresponde para este tipo de filtro.

Por eso, cuando combinemos condiciones en Pandas, vamos a usar siempre esta estructura:

```python
df[(condición_1) & (condición_2)]
```

Podemos verlo como una regla práctica:

```text
cada condición va entre paréntesis
```

Esta costumbre evita muchos errores y hace que el filtro sea más claro de leer.

## Más ejemplos con &

El operador `&` sirve para construir filtros donde todas las condiciones deben cumplirse al mismo tiempo.

Por ejemplo, podemos seleccionar las ventas realizadas en la sucursal `"Centro"` que fueron pagadas con `"Credito"`.

In [ ]:
df[(df["sucursal"] == "Centro") & (df["forma_pago"] == "Credito")]

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03


En este caso, una fila aparece en el resultado solamente si cumple las dos condiciones: la venta debe haberse realizado en la sucursal `"Centro"` y la forma de pago debe ser `"Credito"`.

También podemos combinar una condición categórica con una condición numérica. Por ejemplo, podemos seleccionar los productos de `"Libreria"` cuya cantidad vendida sea mayor o igual que `20`.

In [ ]:
df[(df["categoria"] == "Libreria") & (df["cantidad_vendida"] >= 20)]

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01
3,Regla,Libreria,500,25,Sur,Efectivo,2024-03-02
12,Goma,Libreria,300,30,Sur,Efectivo,2024-03-07
13,Lapiz,Libreria,250,50,Centro,Debito,2024-03-07


Este filtro conserva únicamente las filas donde la categoría es `"Libreria"` y la cantidad vendida es mayor o igual que `20`.

Otro ejemplo posible sería seleccionar las ventas de la sucursal `"Norte"` con precio mayor que `5000`:

In [ ]:
df[(df["sucursal"] == "Norte") & (df["precio"] > 5000)]

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
2,Mochila,Accesorios,8500,5,Norte,Credito,2024-03-02
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05


Estos ejemplos muestran que podemos combinar condiciones sobre columnas de distinto tipo. Una condición puede referirse a una categoría, otra a un número, otra a una forma de pago o a cualquier otra variable del dataset.

La clave es que, cuando usamos `&`, todas las condiciones deben cumplirse al mismo tiempo para que la fila quede incluida en el resultado.

## Combinar condiciones con |

Cuando queremos seleccionar filas que cumplan una condición u otra, usamos el operador `|`. Este operador funciona como un “o lógico”. Una fila queda incluida si cumple al menos una de las condiciones.

Por ejemplo, supongamos que queremos seleccionar las ventas realizadas en la sucursal `"Centro"` o en la sucursal `"Norte"`.

Podemos escribir:

In [ ]:
df[(df["sucursal"] == "Centro") | (df["sucursal"] == "Norte")]

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
0,Cuaderno,Libreria,1200,15,Centro,Efectivo,2024-03-01
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01
2,Mochila,Accesorios,8500,5,Norte,Credito,2024-03-02
4,Cartuchera,Accesorios,3200,8,Norte,Debito,2024-03-03
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
7,Resaltadores,Libreria,2100,12,Centro,Debito,2024-03-04
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06
11,Carpeta,Libreria,950,18,Norte,Efectivo,2024-03-06
13,Lapiz,Libreria,250,50,Centro,Debito,2024-03-07


El resultado incluye las filas donde la sucursal es `"Centro"` y también las filas donde la sucursal es `"Norte"`.

En este caso no hace falta que una fila cumpla ambas condiciones al mismo tiempo. De hecho, una venta no puede pertenecer simultáneamente a la sucursal `"Centro"` y a la sucursal `"Norte"`. Al usar `|`, alcanza con que se cumpla una de las dos condiciones.

Podemos leer la instrucción así:

```text
mostrá las filas donde
sucursal sea Centro
o
sucursal sea Norte
```

También podemos usar `|` con otras columnas. Por ejemplo, podemos seleccionar las ventas pagadas con `"Credito"` o `"Debito"`:

In [ ]:
df[(df["forma_pago"] == "Credito") | (df["forma_pago"] == "Debito")]

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01
2,Mochila,Accesorios,8500,5,Norte,Credito,2024-03-02
4,Cartuchera,Accesorios,3200,8,Norte,Debito,2024-03-03
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
6,Auriculares,Tecnologia,7600,6,Sur,Credito,2024-03-04
7,Resaltadores,Libreria,2100,12,Centro,Debito,2024-03-04
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06
13,Lapiz,Libreria,250,50,Centro,Debito,2024-03-07
14,Agenda,Libreria,4300,9,Sur,Credito,2024-03-08


En este caso, el resultado excluye las ventas pagadas en efectivo y conserva solamente las que fueron pagadas con crédito o débito.

Al igual que con `&`, cada condición debe ir entre paréntesis.

## Diferencia entre & y |

Los operadores `&` y `|` permiten combinar condiciones, pero no significan lo mismo.

Usamos `&` cuando queremos que todas las condiciones se cumplan al mismo tiempo. En cambio, usamos `|` cuando alcanza con que se cumpla al menos una de las condiciones.

Por ejemplo, este filtro usa `&`:

In [ ]:
df[(df["sucursal"] == "Centro") & (df["forma_pago"] == "Credito")]

El resultado incluye solamente las ventas que cumplen ambas condiciones: fueron realizadas en la sucursal `"Centro"` y fueron pagadas con `"Credito"`.

En cambio, este filtro usa `|`:

In [ ]:
df[(df["sucursal"] == "Centro") | (df["forma_pago"] == "Credito")]

El resultado incluye las ventas que cumplen al menos una de las dos condiciones: ventas realizadas en la sucursal `"Centro"`, ventas pagadas con `"Credito"`, o ventas que cumplen ambas condiciones al mismo tiempo.

La diferencia puede resumirse así:

```text
&  → deben cumplirse todas las condiciones
|  → debe cumplirse al menos una condición
```

Esta diferencia es muy importante porque puede cambiar mucho la cantidad de filas obtenidas. En general, los filtros con `&` suelen ser más restrictivos, porque exigen que se cumplan varios criterios al mismo tiempo. Los filtros con `|` suelen ser más amplios, porque aceptan filas que cumplan cualquiera de las condiciones indicadas.

## Negar una condición con ~

A veces no queremos seleccionar las filas que cumplen una condición, sino justamente las que no la cumplen. Para eso usamos el operador `~`, que permite negar una condición.

Por ejemplo, si queremos seleccionar todas las ventas que no corresponden a la sucursal `"Sur"`, podemos escribir:

In [ ]:
df[~(df["sucursal"] == "Sur")]

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
0,Cuaderno,Libreria,1200,15,Centro,Efectivo,2024-03-01
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01
2,Mochila,Accesorios,8500,5,Norte,Credito,2024-03-02
4,Cartuchera,Accesorios,3200,8,Norte,Debito,2024-03-03
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
7,Resaltadores,Libreria,2100,12,Centro,Debito,2024-03-04
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06
11,Carpeta,Libreria,950,18,Norte,Efectivo,2024-03-06
13,Lapiz,Libreria,250,50,Centro,Debito,2024-03-07


La condición original es:

```python
df["sucursal"] == "Sur"
```

Esa condición devuelve `True` para las filas donde la sucursal es `"Sur"`.

Al colocar `~` delante de la condición, invertimos el resultado: los `True` pasan a ser `False` y los `False` pasan a ser `True`.

Entonces:

```python
df[~(df["sucursal"] == "Sur")]
```

significa: mostrá las filas donde la sucursal no sea `"Sur"`.

En este caso, podríamos obtener el mismo resultado usando `!=`:


In [ ]:
df[df["sucursal"] != "Sur"]

Ambas formas son válidas para este ejemplo. Sin embargo, `~` resulta especialmente útil cuando queremos negar condiciones más largas o cuando usamos métodos como `isin()`.

Por ejemplo, podemos seleccionar las ventas que no pertenecen a las categorías `"Tecnologia"` ni `"Accesorios"`:

In [ ]:
df[~(df["categoria"].isin(["Tecnologia", "Accesorios"]))]

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
0,Cuaderno,Libreria,1200,15,Centro,Efectivo,2024-03-01
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01
3,Regla,Libreria,500,25,Sur,Efectivo,2024-03-02
7,Resaltadores,Libreria,2100,12,Centro,Debito,2024-03-04
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05
9,Compas,Libreria,2600,10,Sur,Efectivo,2024-03-05
11,Carpeta,Libreria,950,18,Norte,Efectivo,2024-03-06
12,Goma,Libreria,300,30,Sur,Efectivo,2024-03-07
13,Lapiz,Libreria,250,50,Centro,Debito,2024-03-07
14,Agenda,Libreria,4300,9,Sur,Credito,2024-03-08


En este caso, `isin()` identifica las filas cuya categoría está dentro de la lista `["Tecnologia", "Accesorios"]`. Luego, el operador `~` invierte esa condición y conserva las filas que no pertenecen a esas categorías.

Podemos resumirlo así:

```text
~  → niega una condición
```

El operador `~` debe colocarse delante de una condición entre paréntesis. De esa manera, Pandas sabe exactamente qué condición queremos invertir.


## Guardar condiciones en variables

Cuando combinamos condiciones, las instrucciones pueden volverse largas. Esto no está mal, pero a veces dificulta la lectura.

Por ejemplo, este filtro selecciona ventas de la categoría `"Tecnologia"` con precio mayor que `8000`:

In [ ]:
df[(df["categoria"] == "Tecnologia") & (df["precio"] > 8000)]

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06


La instrucción funciona correctamente, pero podemos escribirla de una manera más clara guardando cada condición en una variable.

Primero creamos las condiciones:

In [ ]:
es_tecnologia = df["categoria"] == "Tecnologia"
precio_mayor_a_8000 = df["precio"] > 8000

Luego combinamos esas variables dentro del filtro:

In [ ]:
df[es_tecnologia & precio_mayor_a_8000]

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06


El resultado es el mismo que antes, pero el código puede leerse con más facilidad.

Ahora podemos interpretar el filtro casi como una frase:

```text
mostrar las filas donde
es_tecnologia
y
precio_mayor_a_8000
```

Esta forma de trabajo es especialmente útil cuando las condiciones tienen nombres claros. Por ejemplo:


In [ ]:
es_centro = df["sucursal"] == "Centro"
pago_con_credito = df["forma_pago"] == "Credito"

df[es_centro & pago_con_credito]

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03


Al guardar condiciones en variables, el código queda más ordenado y resulta más sencillo detectar errores. También se vuelve más fácil modificar una condición sin tener que reescribir todo el filtro.

Esta práctica será cada vez más útil a medida que construyamos consultas más complejas.

## Combinar condiciones y seleccionar columnas

Cuando combinamos condiciones, muchas veces no necesitamos mostrar todas las columnas del `DataFrame`. Tal vez queremos ver solamente las variables relevantes para la pregunta que estamos respondiendo.

Para eso podemos combinar condiciones dentro de `loc`.

La estructura general es:

```python
df.loc[condición, columnas]
```

Por ejemplo, podemos seleccionar las ventas de `"Tecnologia"` con precio mayor que `8000`, pero mostrar solamente el producto, el precio, la sucursal y la forma de pago.

In [ ]:
df.loc[
    (df["categoria"] == "Tecnologia") & (df["precio"] > 8000),
    ["producto", "precio", "sucursal", "forma_pago"]
]

,producto,precio,sucursal,forma_pago
5,Calculadora,18500,Centro,Credito
10,Pendrive,9800,Centro,Debito


Esta instrucción tiene dos partes. En la primera parte indicamos la condición que deben cumplir las filas. En la segunda parte indicamos las columnas que queremos conservar en el resultado.

También podemos usar condiciones guardadas en variables:

In [ ]:
es_tecnologia = df["categoria"] == "Tecnologia"
precio_mayor_a_8000 = df["precio"] > 8000

df.loc[
    es_tecnologia & precio_mayor_a_8000,
    ["producto", "precio", "sucursal", "forma_pago"]
]

,producto,precio,sucursal,forma_pago
5,Calculadora,18500,Centro,Credito
10,Pendrive,9800,Centro,Debito


El resultado es el mismo, pero el código puede resultar más fácil de leer porque las condiciones tienen nombres descriptivos.

Veamos otro ejemplo. Podemos seleccionar las ventas de la sucursal `"Centro"` o `"Norte"` que hayan sido pagadas con `"Debito"`, y mostrar solamente algunas columnas.

In [ ]:
es_centro_o_norte = df["sucursal"].isin(["Centro", "Norte"])
pago_con_debito = df["forma_pago"] == "Debito"

df.loc[
    es_centro_o_norte & pago_con_debito,
    ["producto", "sucursal", "forma_pago", "cantidad_vendida"]
]

Esta forma de trabajo permite construir resultados más claros. No solo filtramos las filas que cumplen ciertos criterios, sino que además elegimos qué información queremos mostrar de esas filas.

Cuando el filtro tiene varias condiciones, usar `loc` junto con variables descriptivas puede hacer que el código sea más legible y más fácil de revisar.

## Errores frecuentes al combinar condiciones

Al combinar condiciones en Pandas, hay algunos errores que aparecen con frecuencia. Conocerlos ayuda a interpretar mejor los mensajes de error y a corregir el código con más seguridad.

Un primer error común es usar `and` en lugar de `&`.

En Python, `and` sirve para combinar valores booleanos simples, como `True` y `False`. Pero cuando trabajamos con columnas de Pandas, cada condición produce una máscara booleana completa, no un único valor verdadero o falso. Por eso, para combinar condiciones sobre columnas usamos `&`.

In [ ]:
# Esta instrucción produciría un error porque en Pandas no usamos and
# para combinar condiciones sobre columnas.

# df[(df["categoria"] == "Tecnologia") and (df["precio"] > 8000)]

La forma correcta es:

In [ ]:
df[(df["categoria"] == "Tecnologia") & (df["precio"] > 8000)]

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06


Otro error frecuente es usar `or` en lugar de `|`. Si queremos seleccionar filas que cumplan una condición u otra, debemos usar `|`.

In [ ]:
# Esta instrucción produciría un error porque en Pandas no usamos or
# para combinar condiciones sobre columnas.

# df[(df["sucursal"] == "Centro") or (df["sucursal"] == "Norte")]

La forma correcta es:

In [ ]:
df[(df["sucursal"] == "Centro") | (df["sucursal"] == "Norte")]

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
0,Cuaderno,Libreria,1200,15,Centro,Efectivo,2024-03-01
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01
2,Mochila,Accesorios,8500,5,Norte,Credito,2024-03-02
4,Cartuchera,Accesorios,3200,8,Norte,Debito,2024-03-03
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
7,Resaltadores,Libreria,2100,12,Centro,Debito,2024-03-04
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06
11,Carpeta,Libreria,950,18,Norte,Efectivo,2024-03-06
13,Lapiz,Libreria,250,50,Centro,Debito,2024-03-07


También es muy común olvidar los paréntesis alrededor de cada condición. En Pandas, cuando combinamos condiciones, cada comparación debe quedar encerrada entre paréntesis.

In [ ]:
# Esta instrucción produciría un error porque faltan paréntesis.

# df[df["categoria"] == "Tecnologia" & df["precio"] > 8000]

La forma correcta es:

In [ ]:
df[(df["categoria"] == "Tecnologia") & (df["precio"] > 8000)]

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06


Finalmente, también debemos tener cuidado con el operador `~`. Este operador niega una condición, pero necesita saber exactamente qué condición debe invertir. Por eso conviene usarlo siempre junto con paréntesis.

In [ ]:
df[~(df["sucursal"] == "Sur")]

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
0,Cuaderno,Libreria,1200,15,Centro,Efectivo,2024-03-01
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01
2,Mochila,Accesorios,8500,5,Norte,Credito,2024-03-02
4,Cartuchera,Accesorios,3200,8,Norte,Debito,2024-03-03
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
7,Resaltadores,Libreria,2100,12,Centro,Debito,2024-03-04
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06
11,Carpeta,Libreria,950,18,Norte,Efectivo,2024-03-06
13,Lapiz,Libreria,250,50,Centro,Debito,2024-03-07


Una buena regla práctica es recordar estas equivalencias:

```text
&   reemplaza a "y" cuando combinamos condiciones de Pandas
|   reemplaza a "o" cuando combinamos condiciones de Pandas
~   niega una condición
```

Y, junto con eso, mantener esta costumbre:

```text
cada condición va entre paréntesis
```

Con estas dos ideas, la mayoría de los filtros combinados se vuelven mucho más fáciles de escribir y revisar.

## Resumen del capítulo

En este capítulo aprendimos a combinar condiciones dentro de un mismo filtro.

Hasta ahora habíamos construido filtros simples, usando una sola condición. Por ejemplo, podíamos seleccionar ventas de una categoría determinada o ventas cuyo precio superara cierto valor. En este capítulo dimos un paso más y comenzamos a construir filtros más precisos, capaces de combinar varios criterios al mismo tiempo.

Primero trabajamos con el operador `&`, que permite expresar una condición del tipo “esto y aquello”. Por ejemplo:

```python
df[(df["categoria"] == "Tecnologia") & (df["precio"] > 8000)]
```

Esta instrucción conserva solamente las filas donde se cumplen ambas condiciones: la categoría debe ser `"Tecnologia"` y el precio debe ser mayor que `8000`.

Luego trabajamos con el operador `|`, que permite expresar una condición del tipo “esto o aquello”. Por ejemplo:

```python
df[(df["sucursal"] == "Centro") | (df["sucursal"] == "Norte")]
```

En este caso, una fila queda incluida si pertenece a la sucursal `"Centro"` o si pertenece a la sucursal `"Norte"`. No hace falta que se cumplan ambas condiciones al mismo tiempo; alcanza con que se cumpla una de ellas.

Después incorporamos el operador `~`, que permite negar una condición. Por ejemplo:

```python
df[~(df["sucursal"] == "Sur")]
```

Esta instrucción selecciona las ventas cuya sucursal no es `"Sur"`.

También vimos que, al combinar condiciones en Pandas, cada condición debe escribirse entre paréntesis. Esta regla es muy importante para evitar errores y para que Python interprete correctamente cada comparación.

Más adelante guardamos condiciones en variables para mejorar la legibilidad del código:

```python
es_tecnologia = df["categoria"] == "Tecnologia"
precio_mayor_a_8000 = df["precio"] > 8000

df[es_tecnologia & precio_mayor_a_8000]
```

Esta forma de trabajo permite construir filtros más claros, especialmente cuando las condiciones empiezan a ser largas o cuando queremos reutilizarlas.

Finalmente combinamos filtros con selección de columnas usando `loc`:

```python
df.loc[
    (df["categoria"] == "Tecnologia") & (df["precio"] > 8000),
    ["producto", "precio", "sucursal", "forma_pago"]
]
```

Esta estructura nos permite seleccionar las filas que cumplen ciertas condiciones y, al mismo tiempo, mostrar solamente las columnas que necesitamos.

La idea principal de este capítulo fue:

```text
Podemos combinar condiciones para construir filtros más precisos sobre un DataFrame.
```

Los operadores lógicos `&`, `|` y `~` son herramientas fundamentales para transformar una pregunta de análisis en una selección concreta de datos.

## Próximo paso

Ya sabemos construir filtros simples y también combinar varias condiciones dentro de un mismo filtro.

El siguiente paso será aprender a ordenar datos. Muchas veces, después de filtrar un dataset, queremos ordenar los resultados: por precio, por cantidad vendida, por fecha, por categoría o por cualquier otra variable relevante.

Ordenar un `DataFrame` nos permitirá identificar valores altos o bajos, revisar rankings, encontrar registros extremos y presentar la información de manera más clara.